#  This Marine Cold Spells (MCS) pipeline detect and calculte the MCS using *10th percentile threshold* which is standered MHW detection *Hobdey et at. (2016)* method.

In [ ]:
# primary requirements are the libraries and packages must be installed (python3, numpy, pandas, xarray, dask, bottleneck, netCDF4, cftime, matplotlib, marineHeatWaves)
# also fixes np.NaN to np.nan in marineHeatWaves module

import sys, platform
import numpy as np
import pandas as pd
import xarray as xr, dask, bottleneck, netCDF4, cftime
from pathlib import Path
from typing import List, Optional, Tuple, Dict, Union
import matplotlib.pyplot as plt
import os
import matplotlib.dates as mdates
import marineHeatWaves as mhw, re, pathlib
from importlib.metadata import version, PackageNotFoundError

# convert np.NAN to np.nan to avoid dependency issues
if not hasattr(np, "NaN"):
    np.NaN = np.nan  

# Check the package is working or not by printing the versions of the dependencies
try:
    mhw_ver = version('marineHeatWaves')
except PackageNotFoundError:
    mhw_ver = getattr(mhw, '__version__', 'unknown')
print('OK:', pd.__version__, xr.__version__, dask.__version__, 'mhw:', mhw_ver)
print('mhw module file:', getattr(mhw, '__file__', 'unknown'))

# check the marineHeatWaves module file and replace np.NaN with np.nan
p = pathlib.Path(mhw.__file__)
p.write_text(re.sub(r'\bnp\.NaN\b', 'np.nan', p.read_text()))

print(sys.executable)
print(platform.python_version())

In [ ]:
#check if coldspells is working in MHW package
t = pd.date_range("2000-01-01", periods=40, freq="D")
ords = np.array([d.toordinal() for d in t], dtype=int)
temp = 25 + np.sin(np.linspace(0, 6.28, 40)); temp[15:25] += 2

res, clim = mhw.detect(ords, temp, climatologyPeriod=[2000, 2000],
                       pctile=10, minDuration=5, joinAcrossGaps=True, coldSpells=True)
print("events:", len(res.get("time_start", [])))

# Verifies that core libs are importable and compatible with marineHeatWaves.
from importlib import import_module
import numpy as np
import marineHeatWaves as mhw

want = ["numpy", "pandas", "xarray", "dask", "bottleneck","netCDF4", "cftime", "marineHeatWaves"]
vers = {}
problems = []

for m in want:
    try:
        import_module(m)
        try:
            vers[m] = version(m)
        except PackageNotFoundError:
            mod = import_module(m)
            vers[m] = getattr(mod, "__version__", "unknown")
    except Exception as e:
        problems.append((m, str(e)))
        vers[m] = "IMPORT FAILED"

print("Package versions detailes:")
for k in want:
    print(f"{k:16s} : {vers[k]}")
if problems:
    print("\n[ENV WARNING] Some packages failed to import:")
    for m, msg in problems:
        print(f" - {m}: {msg}")

# marineHeatWaves.detect accepts ordinal ints + 1D temp array
try:
    # make a synthetic 40-day series with a warm spell
    days = np.array([pd.Timestamp("2000-01-01") + pd.Timedelta(i, "D") for i in range(40)])
    ords = np.array([d.toordinal() for d in days], dtype=int)
    temp = 25 + np.sin(np.linspace(0, 6.28, 40))
    temp[15:25] += 2.0  # warm event
    res, clim = mhw.detect(ords, temp, climatologyPeriod=[1991, 2020], pctile=90, minDuration=5, joinAcrossGaps=True, coldSpells= True)
    print("\nmarineHeatWaves.detect is working correctly.")
    print(f"Detected events: {len(res.get('time_start', []))}")
except Exception as e:
    print("\n[ENV ERROR] marineHeatWaves.detect detection failed:", e)

**Common code for Climatologoy, Threshold, Mean STT , SST Trend and MHW days correlation analysis**

In [ ]:
# Common code for Climatologoy, Threshold plots and analysis
#Run this code first before running other analysis codes

from glob import glob
from pathlib import Path
from typing import Dict, Tuple, Optional
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import datetime as dt
import matplotlib.dates as mdates
import marineHeatWaves as mhw  

# convert np.NAN to np.nan to avoid dependency issues
import numpy as _np
if not hasattr(_np, "NaN"):
    _np.NaN = _np.nan

# File path
FILES_GLOB = "/home/Desktop/Noah_data_1982-2024_SST_daily_mean/sst.day.mean.*.nc"
VAR = "sst"
CLIM_YEARS: Tuple[int,int] = (1982, 2024)   
REGIONS: Dict[str, Dict[str, float]] = {   # modify the lat-lon bounds as per your region
    "Arabian Sea":    {"lon_min": 40.0, "lon_max": 78.0,  "lat_min": 0.0, "lat_max": 30.0},
    "Bay Of Bengal":   {"lon_min": 78.0, "lon_max": 110.0, "lat_min": 0.0, "lat_max": 30.0},
    "North Indian Ocean": {"lon_min": 40.0, "lon_max": 110.0, "lat_min": 0.0, "lat_max": 30.0},
}

SEASONS = {"DJF": [12, 1, 2],"MAM": [3, 4, 5],"JJA": [6, 7, 8],"SON": [9, 10, 11],}
season_order = ["DJF", "MAM", "JJA", "SON"] 
OUTDIR = Path("outputs"); OUTDIR.mkdir(parents=True, exist_ok=True)

# leap-aware reference year 
_ref = pd.date_range("2000-01-01", "2000-12-31", freq="D") 
_doy_to_month = pd.Series(_ref.month.values, index=np.arange(1, 367))
_month_to_doy0 = {m: (_doy_to_month[_doy_to_month == m].index.values - 1) for m in range(1, 13)}


def open_mfdataset(paths_glob: str, chunks={"time": 120}, engine: str = "netcdf4") -> xr.Dataset:
    paths = sorted(glob(paths_glob))
    if not paths:
        raise FileNotFoundError(f"No files match: {paths_glob}")
    print(f"[open] {len(paths)} files")
    return xr.open_mfdataset(paths, combine="by_coords", parallel=True, chunks=chunks, engine=engine)

def subset_box(ds: xr.Dataset, box: Dict[str, float]) -> xr.Dataset:
    latn = "lat" if "lat" in ds.coords else "latitude"
    lonn = "lon" if "lon" in ds.coords else "longitude"
    return ds.sel({latn: slice(box["lat_min"], box["lat_max"]),lonn: slice(box["lon_min"], box["lon_max"])})

def area_weighted_boxmean(da: xr.DataArray) -> xr.DataArray:
    latn = "lat" if "lat" in da.coords else "latitude"
    w = np.cos(np.deg2rad(da[latn]))
    return da.weighted(w).mean(dim=[latn, "lon" if "lon" in da.coords else "longitude"])

In [ ]:
# Long-term Climatology, 90th and 10th percentile Threshold of Arabian Sea, Bay Of Bengal, North Indian Ocean

def mhw_seas_thresh_doy(da_box: xr.DataArray, clim_years: Tuple[int,int]) -> Tuple[np.ndarray, np.ndarray]:

    t_index = da_box["time"].to_index()

    # safety-clip baseline to data span
    y0, y1 = max(clim_years[0], t_index.year.min()), min(clim_years[1], t_index.year.max())

    ords = np.array([d.toordinal() for d in t_index], dtype=int)
    temp = da_box.values.astype(float)

    # clim['seas'] and clim['thresh'] for the 90th percentile 
    res_90th, clim_90th = mhw.detect(ords, temp, climatologyPeriod=[int(y0), int(y1)], pctile=90, minDuration=5, joinAcrossGaps=True)
    seas_full   = np.asarray(clim_90th["seas"])
    thresh_90th_full = np.asarray(clim_90th["thresh"])

    # clim['seas'] and clim['thresh'] for the 10th percentile 
    res_10th, clim_10th = mhw.detect(ords, temp, climatologyPeriod=[int(y0), int(y1)], pctile=10, minDuration=5, joinAcrossGaps=True)
    thresh_10th_full = np.asarray(clim_10th["thresh"])

    # Group by day-of-year to get 366-length curves (leap-aware).
    doy = t_index.dayofyear.values
    seas_366   = np.full(366, np.nan, float)
    thresh_90th_366 = np.full(366, np.nan, float)
    thresh_10th_366 = np.full(366, np.nan, float)
    for d in range(1, 367): 
        m = (doy == d)
        if m.any():
            seas_366[d-1]   = np.nanmean(seas_full[m])
            thresh_90th_366[d-1] = np.nanmean(thresh_90th_full[m])
            thresh_10th_366[d-1] = np.nanmean(thresh_10th_full[m])
    return seas_366, thresh_90th_366, thresh_10th_366

# Load dataset
ds_annual= open_mfdataset(FILES_GLOB)
# Compute curves per region and store
curves_annual = {}  
for name, box in REGIONS.items():
    da = subset_box(ds_annual[[VAR]], box)[VAR]
    da_box = area_weighted_boxmean(da)
    seas_doy, thresh_90th_doy, thresh_10th_doy = mhw_seas_thresh_doy(da_box, CLIM_YEARS)
    curves_annual[name] = {"seas": seas_doy, "90th thresh": thresh_90th_doy, "10th thresh": thresh_10th_doy}

# Plotting three stacked panels with shared X-axis
x_dates = pd.date_range("2000-01-01", "2000-12-31", freq="D")  
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(11, 8), sharex=True, constrained_layout=True)

# use common y-limits for comparability
all_vals = np.concatenate([np.r_[v["seas"], v["90th thresh"], v["10th thresh"]] for v in curves_annual.values()])
ymin = float(np.nanmin(all_vals)) - 0.2
ymax = float(np.nanmax(all_vals)) + 0.2 

for ax, (name, ct) in zip(axes, curves_annual.items()):
    ax.plot(x_dates, ct["seas"],   label="Climatology (Annual)", color="C0", lw=1.6)
    ax.plot(x_dates, ct["90th thresh"], label="90th perc Threshold", ls="--", color= "#ff7f0e", lw=1.6)
    ax.plot(x_dates, ct["10th thresh"], label="10th perc Threshold", ls="--", color= "#24fa11", lw=1.6)
    ax.set_ylabel("Temp (°C)", fontweight="bold")
    ax.set_title(f"{name}: Long- Term Climatology, 90th and 10th perc Threshold ({CLIM_YEARS[0]}–{CLIM_YEARS[1]})", fontweight="bold")
    ax.legend(loc="upper left")
    ax.set_ylim(ymin, ymax)

year = 2000  
tick_dates = []
for m in range(1, 13):
    tick_dates.append(pd.Timestamp(year, m, 1))
    tick_dates.append(pd.Timestamp(year, m, 15))
tick_dates.append(pd.Timestamp(year, 12, 31))  

axes[-1].set_xticks(tick_dates)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
axes[-1].set_xlim(x_dates[0], x_dates[-1])
axes[-1].set_xlabel("Day of Year", fontweight="bold")
fig.autofmt_xdate()
